# Atmospheric Scattering

A geometrically unobstructed line of sight is not enough for the target to be visible: air scatters light, so a distant target fades into the background sky. This notebook assembles the **contrast** of the target against the sky, following Michael Vollmer, *Below the horizon — the physics of extreme visual ranges* (2020). The target is considered visible when its contrast exceeds $0.02$, the threshold of human vision. This is computed in `LineOfSight.get_contrast` and `LineOfSight.has_contrast` in `scripts/commons.py`.

In polar coordinates the observer is at $P_1 = (R_\oplus + h_1,\, 0)$ and the target at $P_2 = (R_\oplus + h_2,\, D_s/R_\oplus)$. Light follows the arc of a circle of radius $R_L = k R_\oplus$ (see `atmospheric_refraction.ipynb`).

**Notation**

- $C$ — contrast of the target against the sky.
- $h_1, h_2$ — elevation of the observer and target.
- $D_s$ — surface (sea-level arc) distance between observer and target.
- $D_d$ — direct (straight-line) distance between observer and target.
- $H$ — scale height of the atmosphere.
- $\beta_0$ — scattering coefficient at sea level.
- $\beta(h) = \beta_0\, e^{-h/H}$ — scattering coefficient at elevation $h$.
- $\beta_1, \beta_2$ — average scattering coefficient over the shaded and sunlit segments.
- $S$ — shaded ratio: fraction of the path in shadow, starting at the observer.
- $a$ — shade irradiation ratio: brightness of the shaded segment relative to the sunlit segment.
- $x$ — surface distance from the observer; $\theta = x/R_\oplus$, $\phi = D_s/R_\oplus$.
- $L(x)$ — distance from the centre of the Earth to the light arc at $x$.
- $h(x) = L(x) - R_\oplus$ — height of the light arc above sea level at $x$.
- $(M_x, M_y)$ — centre of the light-arc circle in Cartesian coordinates.

---

**Contrast** (Vollmer eq. 9). The path is split into a shaded segment of length $d_1 = S D_s$ (starting at the observer) and a sunlit segment of length $d_2 = (1 - S) D_s$:

$$C = \frac{e^{-\beta_2 d_2}}{1 - a + \left( \dfrac{a}{e^{-\beta_1 d_1}} \right)}$$

**Average scattering coefficients** (Vollmer eq. 7). Scattering thins with altitude, so the sea-level coefficient is weighted by the height of the light path and integrated along it. The integrals run over the light path, with arc-length element $ds$:

$$\beta_1 = \frac{\beta_0}{d_1}\int_{0}^{d_1} e^{-h(x)/H}\,ds, \qquad \beta_2 = \frac{\beta_0}{d_2}\int_{d_1}^{D_s} e^{-h(x)/H}\,ds$$
$$d_1 = S D_s, \qquad d_2 = (1 - S) D_s, \qquad h(x) = L(x) - R_\oplus$$

Here $ds$ is the arc-length element of the light path. Parameterising by surface distance $x$ (so $\theta = x/R_\oplus$), $ds = \sqrt{(L/R_\oplus)^2 + (dL/dx)^2}\,dx \approx (L/R_\oplus)\,dx$, so in code the surface-distance integral is weighted by $L/R_\oplus$ (see `LineOfSight.get_contrast`).

**Height of the light arc.** With $\theta = x/R_\oplus$, the light circle in polar form is

$$L(x) = M_x\cos\theta + M_y\sin\theta + \sqrt{(M_x\cos\theta + M_y\sin\theta)^2 + R_L^2 - M_x^2 - M_y^2}$$

where the centre of the light circle and the endpoints are

$$M_x = \frac{x_1 + x_2}{2} - Z\,\frac{y_2 - y_1}{D_d}, \qquad M_y = \frac{y_1 + y_2}{2} + Z\,\frac{x_2 - x_1}{D_d}, \qquad Z = \sqrt{R_L^2 - \left(\frac{D_d}{2}\right)^2}$$
$$P_1 = (x_1, y_1) = (R_\oplus + h_1,\; 0), \qquad P_2 = (x_2, y_2) = \big((R_\oplus + h_2)\cos\phi,\; (R_\oplus + h_2)\sin\phi\big)$$
$$D_d = \sqrt{(x_2 - x_1)^2 + (y_2 - y_1)^2}, \qquad \phi = \frac{D_s}{R_\oplus}$$

**Choosing $S$ and $a$.** The shaded segment is the near, observer-side half of the path ($S = 0.5$), because airlight is dominated by the air closest to the observer, so shadowing it helps contrast most. The shade irradiation ratio $a$ is set per line of sight from its bearing, to approximate sunrise/sunset. The sun can only rise or set along the east-west horizon, so only an east-west sightline can place the observer's foreground in shadow with the target lit beyond; a north-south sightline has the sun broadside and gets no benefit. Hence $a = 1 - (1 - a_{\min})\,|\sin(\text{bearing})|$ with $a_{\min} = 0.1$: east-west gives $a = 0.1$ (deepest shadow, highest contrast), north-south gives $a = 1$ (plain extinction), and bearings between interpolate. $a_{\min}$ stays above $0$ because fully shadowed air is still lit by diffuse skylight.
